<a href="https://colab.research.google.com/github/halilcankaroglu/yzm511-midterm-mlp/blob/main/01_eda_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/halilcankaroglu/yzm511-midterm-mlp.git
%cd yzm511-midterm-mlp

Cloning into 'yzm511-midterm-mlp'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 60.12 KiB | 2.50 MiB/s, done.
/content/yzm511-midterm-mlp


In [ ]:
import pandas as pd

df = pd.read_csv("data/bank-additional.csv", sep=";")
print(df.shape)                  # beklenen: (4119, 21)
print(df.dtypes)
print(df["y"].value_counts())

df = df.drop(columns=["duration"])
print(df.duplicated().sum())

(4119, 21)
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object
y
no     3668
yes     451
Name: count, dtype: int64
19


In [ ]:
import os, json
import numpy as np

STUDENT_LAST5 = "29016"
BASE = int(STUDENT_LAST5 + "2026")   # 290162026

SEEDS = {
    "base": BASE,
    "split": BASE + 1,
    "outer_cv": BASE + 2,
    "inner_cv": BASE + 3,
    "early_stop": BASE + 4,
    "random_search": BASE + 5,
    "init": [BASE + 10, BASE + 11, BASE + 12],   # ilki ana model
}
os.makedirs("configs", exist_ok=True)
json.dump(SEEDS, open("configs/seeds.json", "w"), indent=2)
SEEDS

In [ ]:
from sklearn.model_selection import train_test_split

y = (df["y"] == "yes").astype(int)
idx = np.arange(len(df))
dev_idx, test_idx = train_test_split(
    idx, test_size=0.20, stratify=y, random_state=SEEDS["split"])

split = pd.DataFrame({"row_index": idx, "set": "dev", "y": y.values})
split.loc[test_idx, "set"] = "test"
os.makedirs("results", exist_ok=True)
split.to_csv("results/dataset_split.csv", index=False)

split.groupby("set")["y"].agg(["count", "sum", "mean"])

In [ ]:
X_cols = [c for c in df.columns if c != "y"]
dev_keys = set(map(tuple, df.iloc[dev_idx][X_cols].values))
n_twin = sum(tuple(r) in dev_keys for r in df.iloc[test_idx][X_cols].values)
print("Development'ta ikizi olan test satırı:", n_twin)

In [3]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
!git config user.name "Halil Cankaroğlu"
!git config user.email "GITHUB_EPOSTAN"
!git remote set-url origin https://{token}@github.com/halilcankaroglu/yzm511-midterm-mlp.git
!git pull --rebase
!git add configs/seeds.json results/dataset_split.csv
!git commit -m "Seed politikası ve stratified %80/%20 split"
!git push

Already up to date.
fatal: pathspec 'configs/seeds.json' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
